In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path().resolve().parent
CACHE_DIR = PROJECT_ROOT / "data" / "raw"

CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["KAGGLEHUB_CACHE"] = str(CACHE_DIR)

import kagglehub

path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")
print(path)

In [ ]:
from pathlib import Path
import pandas as pd
import os

path = Path(path)

df_news = pd.read_csv(path / "WELFake_Dataset.csv",index_col=0)

In [ ]:
df_news.head()

In [ ]:
df_news = df_news.drop(columns=["title"])

In [ ]:
df_news

In [ ]:
df_news.isnull().sum()

In [ ]:
df_news = df_news.dropna()

In [ ]:
df_news.duplicated().sum()

In [ ]:
df_news = df_news.drop_duplicates()

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

nltk.download('wordnet', "/kaggle/working/nltk_data/")
nltk.download('omw-1.4', "/kaggle/working/nltk_data/")
nltk.download('punkt_tab', "/kaggle/working/nltk_data/")
nltk.download('stopwords', "/kaggle/working/nltk_data/")

In [ ]:
import zipfile
import os

zip_paths = [
    "/kaggle/working/nltk_data/corpora/wordnet.zip",
    "/kaggle/working/nltk_data/corpora/omw-1.4.zip",
    "/kaggle/working/nltk_data/tokenizers/punkt_tab.zip",
    "/kaggle/working/nltk_data/corpora/stopwords.zip"
]

extract_dir = "/kaggle/working/nltk_data/corpora"

os.makedirs(extract_dir, exist_ok=True)

for z in zip_paths:
    with zipfile.ZipFile(z, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
nltk.data.path.append("/kaggle/working/nltk_data/")

In [ ]:
def process_text(text):
    text = re.sub(
        r"\s+", " ", text, flags=re.I
    )  # Remove extra white space from text

    text = re.sub(
        r"\W", " ", str(text)
    )  # Remove all the special characters from text

    text = re.sub(
        r"\s+[a-zA-Z]\s+", " ", text
    )  # Remove all single characters from text

    text = re.sub(
        r"[^a-zA-Z\s]", "", text
    )  # Remove any character that isn't alphabetical

    text = text.lower()

    words = word_tokenize(text)

    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]

    stop_words = set(stopwords.words("english"))
    Words = [word for word in words if word not in stop_words]

    Words = [word for word in Words if len(word) > 3]

    indices = np.unique(Words, return_index=True)[1]
    cleaned_text = np.array(Words)[np.sort(indices)].tolist()

    return cleaned_text

In [ ]:
df_news.to_csv("newsdb2.csv")

In [ ]:
x = df_news.drop("label", axis=1)
y = df_news.label

In [ ]:
texts = list(x["text"])

In [ ]:
cleaned_text = [process_text(text) for text in texts]

In [ ]:
print(cleaned_text[:5])

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    cleaned_text, y, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

tokenizer = Tokenizer()
tokenizer.fit_on_texts(x_train)
word_idx = tokenizer.word_index
v = len(word_idx)
print("the size of vocab =", v)
with open("tokenizer2.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

maxlen = 200
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

In [ ]:
y.value_counts()

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    GlobalMaxPooling1D,
    Dropout,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
maxlen = 200
vocab_size = v + 1
learning_rate = 0.0001

inputs = Input(shape=(maxlen,))
x = Embedding(vocab_size, 100)(inputs)
x = Dropout(0.5)(x)
x = LSTM(150, return_sequences=True)(x)
x = Dropout(0.5)(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.5)(x)
outputs = Dense(2, activation="softmax")(x)

model = Model(inputs, outputs)

optimizer = Adam(learning_rate=learning_rate)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder2 = LabelEncoder()
y_train_encoded = label_encoder2.fit_transform(y_train)
y_test_encoded = label_encoder2.transform(y_test)

with open("label_encoder2.pkl", "wb") as f:
    pickle.dump(label_encoder2, f)

In [ ]:
import tensorflow as tf

y_train_one_hot = tf.keras.utils.to_categorical(y_train_encoded)
y_test_one_hot = tf.keras.utils.to_categorical(y_test_encoded)

In [ ]:
checkpoint = ModelCheckpoint(
    filepath="model2.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
)

history = model.fit(
    x_train,
    y_train_one_hot,
    epochs=10,
    validation_data=(x_test, y_test_one_hot),
    callbacks=[checkpoint],
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Model accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train", "Test"], loc="upper left")
plt.show()

plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Model loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train", "Test"], loc="upper left")
plt.show()

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test_one_hot)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np


y_pred_probs = model.predict(x_test)
y_pred_labels = np.argmax(y_pred_probs, axis=1)
y_true_labels = np.argmax(y_test_one_hot, axis=1)
conf_matrix = confusion_matrix(y_true_labels, y_pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"],
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

maxlen = 200

with open("tokenizer2.pkl", "rb") as f:
    tokenizer = pickle.load(f)

model = tf.keras.models.load_model("model2.keras")

with open("label_encoder2.pkl", "rb") as f:
    label_encoder2 = pickle.load(f)

In [ ]:
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv("fake_or_real_news.csv")

X_raw = df["text"].astype(str).tolist()

label_map = {"fake": "0", "real": "1"}
y_int = df["label"].str.lower().map(label_map).astype(int).values

processed_docs = [" ".join(process_text(t)) for t in X_raw]

with open("tokenizer2.pkl", "rb") as f:
    tokenizer = pickle.load(f)

seqs = tokenizer.texts_to_sequences(processed_docs)
X = pad_sequences(seqs, maxlen=maxlen)

y_one_hot = tf.keras.utils.to_categorical(y_int, num_classes=2)

model = load_model("model2.keras")

loss, acc = model.evaluate(X, y_one_hot, verbose=0)
print("CSV accuracy:", acc)

In [ ]:
from lime.lime_text import LimeTextExplainer
import pickle
import pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv("fake_or_real_news.csv")
texts = df["text"].astype(str).tolist()

with open("tokenizer2.pkl", "rb") as f:
    tokenizer = pickle.load(f)

model = load_model("model2.keras")


def predict_fn(text_list):
    processed = [" ".join(process_text(t)) for t in text_list]
    seqs = tokenizer.texts_to_sequences(processed)
    X = pad_sequences(seqs, maxlen=maxlen)
    return model.predict(X)


explainer = LimeTextExplainer(class_names=["fake", "real"])

sample = texts[0]
exp = explainer.explain_instance(sample, predict_fn, num_features=10)
print(sample)
print(exp.as_list())

In [ ]:
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv("fake_or_real_news.csv").dropna()
label_map = {"fake": 0, "real": 1}
y_new = df["label"].str.lower().map(label_map).astype(int).values
texts = df["text"].astype(str).tolist()

processed = [" ".join(process_text(t)) for t in texts]

with open("tokenizer2.pkl", "rb") as f:
    tokenizer = pickle.load(f)

seqs = tokenizer.texts_to_sequences(processed)
X_new = pad_sequences(seqs, maxlen=maxlen)

y_new_onehot = tf.keras.utils.to_categorical(y_new, num_classes=2)

model = load_model("model2.keras")
model.fit(X_new, y_new_onehot, epochs=15, batch_size=64, validation_split=0.1)
model.save("model2.1.keras")

In [ ]:
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML
import pickle
import pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv("fake_or_real_news.csv")
texts = df["text"].astype(str).tolist()
labels = df["label"].astype(str).tolist()

with open("tokenizer2.pkl", "rb") as f:
    tokenizer = pickle.load(f)

model = load_model("model2.1.keras")


def predict_fn(text_list):
    processed = [" ".join(process_text(t)) for t in text_list]
    seqs = tokenizer.texts_to_sequences(processed)
    X = pad_sequences(seqs, maxlen=maxlen)
    return model.predict(X)


explainer = LimeTextExplainer(class_names=["fake", "real"])
i = 4422
sample = texts[i]
label = labels[i]
exp = explainer.explain_instance(sample, predict_fn, num_features=100)

print(label)
display(HTML(exp.as_html()))

In [ ]:
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_raw = df_news["text"].astype(str).tolist()

y_int = df_news["label"]

processed_docs = [" ".join(process_text(t)) for t in X_raw] 

with open("tokenizer2.pkl", "rb") as f:
    tokenizer = pickle.load(f)

seqs = tokenizer.texts_to_sequences(processed_docs)
X = pad_sequences(seqs, maxlen=maxlen)

y_one_hot = tf.keras.utils.to_categorical(y_int, num_classes=2)

model = load_model("model2.1.keras")

loss, acc = model.evaluate(X, y_one_hot, verbose=0)
print("CSV accuracy:", acc)

In [ ]:
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

with open("tokenizer2.pkl", "rb") as f:
    tokenizer = pickle.load(f)

model = load_model("model2.1.keras")
maxlen = 200


def process_text(t):
    return t.lower().split()


def predict_fn(text_list):
    processed = [" ".join(process_text(t)) for t in text_list]
    seqs = tokenizer.texts_to_sequences(processed)
    X = pad_sequences(seqs, maxlen=maxlen)
    preds = model.predict(X)
    return np.array(preds, dtype=float)


explainer = LimeTextExplainer(class_names=["fake", "real"])

user_text = """Application to DJIA Daily Return Data
We apply the two-component symmetric stable mixture model (with common location μ) to the
daily returns of the 30 stocks in the DJIA dataset. This model is motivated by the stylized fact
that asset returns exhibit high kurtosis and volatility clustering, often effectively approximated
by a mixture of a ”quiet” regime and a ”turbulent” regime. A representative fit is shown in
Figure 9.
Numerical Challenges in Mixture Estimation. While the majority of series yielded
interpretable estimates, several optimizations encountered numerical difficulties: These are
consistent with known pathologies in mixture estimation:
• Singularities: The likelihood function for mixture models is unbounded. If a compo-
nent’s scale parameter σj → 0 while centering on a specific observation (i.e., μ → xk),
the likelihood diverges to infinity. This ”black hole” effect requires careful constraints or
penalized likelihood methods to avoid.
• Flat Likelihoods near Gaussianity: As α → 2, the stable distribution approaches
the Normal. If the data is not sufficiently heavy-tailed, α1 and α2 may both drift toward
2, making the components difficult to distinguish and causing the Hessian to become
ill-conditioned.
• Local Maxima: The mixture likelihood surface is often rugged with multiple local
maxima. The optimizer may converge to a suboptimal solution depending on the starting
values."""

exp = explainer.explain_instance(user_text, predict_fn, num_features=100)

probs = predict_fn([user_text])[0]
pred_idx = int(np.argmax(probs))
pred_label = explainer.class_names[pred_idx]
print("Predicted label:", pred_label, "| Probabilities:", probs)

display(HTML(exp.as_html()))